# 02 — Stage 2: QLoRA Fine-tuning

Fine-tune Mistral-7B-Instruct on the labelled JTBD dataset using QLoRA (4-bit NF4 quantisation + LoRA).
Then evaluate on the test set and compare with Stage 0 and Stage 1 baselines.

**GPU recommendation**: A100 (40 GB) preferred. T4 (16 GB) works with the settings below (~8 GB peak).
Request in: Runtime → Change runtime type → A100 (Colab Pro).

**Estimated training time**:
- T4 (~780 examples, 3 epochs): ~40 min
- A100: ~12 min

In [ ]:
# ── 0. Environment setup ───────────────────────────────────────────────────────
import sys, os

try:
    import google.colab
    IN_COLAB = True
    from google.colab import userdata
    print('Running on Google Colab')
except ImportError:
    IN_COLAB = False
    print('Running locally')

REPO_URL  = 'https://github.com/mikhailarutyunov/text-to-lora-demo.git'
REPO_NAME = 'text-to-lora-demo'

if IN_COLAB:
    if not os.path.exists(f'/content/{REPO_NAME}'):
        !git clone {REPO_URL} /content/{REPO_NAME}
    else:
        !cd /content/{REPO_NAME} && git pull
    %cd /content/{REPO_NAME}

    # Install from requirements file (torch pre-installed on Colab — excluded)
    !pip install -q -r requirements_colab.txt

    HF_TOKEN = userdata.get('HF_TOKEN')
else:
    repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    os.chdir(repo_root)
    from dotenv import load_dotenv; load_dotenv()
    HF_TOKEN = os.environ.get('HF_TOKEN')

if 'src' not in sys.path:
    sys.path.insert(0, 'src')

print(f'Working directory: {os.getcwd()}')

In [ ]:
# ── 1. GPU check ───────────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), 'No GPU found. Change runtime type to GPU.'
print(f'Device: {torch.cuda.get_device_name(0)}')
print(f'VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 2. Hyperparameters (edit here) ────────────────────────────────────────────
CFG = {
    # Model
    'base_model':       'mistralai/Mistral-7B-Instruct-v0.2',
    # LoRA
    'lora_r':           8,       # rank — match T2L default for fair comparison
    'lora_alpha':       16,
    'lora_dropout':     0.05,
    'lora_target':      ['q_proj', 'v_proj'],  # match T2L target modules
    # Training
    'num_epochs':       3,
    'batch_size':       4,       # per-device; increase to 8 on A100
    'grad_accum':       4,       # effective batch = batch_size * grad_accum
    'lr':               2e-4,
    'max_seq_len':      512,
    'warmup_ratio':     0.05,
    'weight_decay':     0.01,
    # Output
    'output_dir':       'checkpoints/jtbd-lora',
    'eval_steps':       50,
    'save_steps':       100,
    'seed':             42,
}
print('Config:', CFG)

In [ ]:
# ── 3. Load dataset ────────────────────────────────────────────────────────────
from data_utils import load_splits, to_hf_dataset, NODE_TYPES

splits = load_splits('data/processed')
train_ds = to_hf_dataset(splits['train'])
val_ds   = to_hf_dataset(splits['val'])
test_ds  = to_hf_dataset(splits['test'])

print(f'Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}')
print('\nSample prompt_with_answer:')
print(train_ds[0]['prompt_with_answer'])

In [ ]:
# ── 4. Load model in 4-bit (QLoRA) ────────────────────────────────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f'Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(CFG['base_model'], token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print(f'Loading model in 4-bit...')
model = AutoModelForCausalLM.from_pretrained(
    CFG['base_model'],
    quantization_config=bnb_config,
    device_map='auto',
    token=HF_TOKEN,
)
model.config.use_cache = False
model.config.pretraining_tp = 1
print('Model loaded.')

In [ ]:
# ── 5. Apply LoRA with PEFT ───────────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=CFG['lora_r'],
    lora_alpha=CFG['lora_alpha'],
    target_modules=CFG['lora_target'],
    lora_dropout=CFG['lora_dropout'],
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# ── 6. Tokenise dataset ───────────────────────────────────────────────────────
def tokenize(example):
    out = tokenizer(
        example['prompt_with_answer'],
        truncation=True,
        max_length=CFG['max_seq_len'],
        padding='max_length',
    )
    labels = out['input_ids'].copy()
    labels = [-100 if tok == tokenizer.pad_token_id else tok for tok in labels]
    out['labels'] = labels
    return out

print('Tokenising...')
train_tok = train_ds.map(tokenize, remove_columns=train_ds.column_names, desc='train')
val_tok   = val_ds.map(tokenize,   remove_columns=val_ds.column_names,   desc='val')
train_tok.set_format('torch')
val_tok.set_format('torch')
print(f'Train tokens shape: {train_tok[0]["input_ids"].shape}')

In [ ]:
# ── 7. Training ───────────────────────────────────────────────────────────────
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq

training_args = TrainingArguments(
    output_dir=CFG['output_dir'],
    num_train_epochs=CFG['num_epochs'],
    per_device_train_batch_size=CFG['batch_size'],
    per_device_eval_batch_size=CFG['batch_size'],
    gradient_accumulation_steps=CFG['grad_accum'],
    learning_rate=CFG['lr'],
    warmup_ratio=CFG['warmup_ratio'],
    weight_decay=CFG['weight_decay'],
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    evaluation_strategy='steps',
    eval_steps=CFG['eval_steps'],
    save_steps=CFG['save_steps'],
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_steps=20,
    report_to='none',
    seed=CFG['seed'],
    gradient_checkpointing=True,
    optim='paged_adamw_8bit',
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print('Starting training...')
trainer.train()

In [ ]:
# ── 8. Save adapter ───────────────────────────────────────────────────────────
model.save_pretrained(CFG['output_dir'] + '/final')
tokenizer.save_pretrained(CFG['output_dir'] + '/final')
print(f'Adapter saved to {CFG["output_dir"]}/final')

In [ ]:
# ── 9. Stage 2 evaluation on test set ────────────────────────────────────────
from data_utils import format_prompt, extract_label_from_response
from tqdm.auto import tqdm
from collections import Counter

model.eval()

@torch.no_grad()
def predict_label(prompt: str, max_new_tokens: int = 12) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return extract_label_from_response(raw)

test_examples = splits['test']
stage2_preds, stage2_labels = [], []
for ex in tqdm(test_examples, desc='Stage 2 eval'):
    pred = predict_label(format_prompt(ex))
    stage2_preds.append(pred)
    stage2_labels.append(ex['node_type'])

acc2 = sum(p == l for p, l in zip(stage2_preds, stage2_labels)) / len(stage2_labels)
print(f'\nStage 2 accuracy: {acc2:.3f}')

In [ ]:
# ── 10. Final comparison ──────────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print('=== Stage 2: Fine-tuned LoRA — Classification Report ===')
print(classification_report(stage2_labels, stage2_preds, labels=NODE_TYPES, zero_division=0))

cm2 = confusion_matrix(stage2_labels, stage2_preds, labels=NODE_TYPES)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(pd.DataFrame(cm2, index=NODE_TYPES, columns=NODE_TYPES),
            annot=True, fmt='d', cmap='Greens', ax=ax)
ax.set_title('Stage 2: Fine-tuned LoRA — confusion matrix')
ax.set_ylabel('True')
ax.set_xlabel('Predicted')
plt.tight_layout()
plt.show()

summary = pd.DataFrame({
    'Stage': ['Stage 0 (zero-shot)', 'Stage 1 (T2L zero-shot)', 'Stage 2 (fine-tuned LoRA)'],
    'Accuracy':  [None, None, acc2],
    'Macro F1':  [None, None, f1_score(stage2_labels, stage2_preds, labels=NODE_TYPES, average='macro', zero_division=0)],
})
print('\n=== Three-Stage Comparison ===')
print(summary.to_string(index=False))